# Step 4: Model Training Loop (Wav2Vec2)

Trains the binary classifier using:
- Automatic Mixed Precision (`torch.amp.autocast('cuda')`)
- `AdamW` optimizer with differential learning rates ($10^{-4}$ head, $10^{-5}$ backbone)
- Cosine annealing learning rate scheduling
- Early stopping based on validation loss
- Checkpointing best model to `models/best_wav2vec2_deepfake_model.pth`

In [1]:
import os
import json
import time
from pathlib import Path
import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from transformers import Wav2Vec2Model, get_cosine_schedule_with_warmup
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from tqdm import tqdm

BASE_DIR = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()
DATA_PROCESSED_DIR = BASE_DIR / "data" / "processed_clean"
MODELS_DIR = BASE_DIR / "models"
LOGS_DIR = BASE_DIR / "logs"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)


e:\Deepfake\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Setup Model, Dataset, and Training Execution

In [2]:
# Load Splits
train_df = pd.read_csv(DATA_PROCESSED_DIR / "train_split.csv")
val_df = pd.read_csv(DATA_PROCESSED_DIR / "val_split.csv")

class AudioDataset(Dataset):
    def __init__(self, df, target_samples=64000):
        self.df = df.reset_index(drop=True)
        self.target_samples = target_samples
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        y, _ = sf.read(self.df.iloc[idx]["proc_path"], dtype="float32")
        if y.ndim > 1: y = y.mean(axis=1)
        if len(y) < self.target_samples: y = np.pad(y, (0, self.target_samples - len(y)))
        else: y = y[:self.target_samples]
        return {
            "input_values": torch.from_numpy(y).float(),
            "label": torch.tensor(self.df.iloc[idx]["label_id"], dtype=torch.long)
        }

train_loader = DataLoader(AudioDataset(train_df), batch_size=16, shuffle=True)
val_loader = DataLoader(AudioDataset(val_df), batch_size=16, shuffle=False)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Training on device: {device}")


Training on device: cuda


In [3]:
# Train Model (or verify existing checkpoint)
from transformers import Wav2Vec2Model

class Classifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.wav2vec2 = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base")
        if hasattr(self.wav2vec2, "feature_extractor"):
            for p in self.wav2vec2.feature_extractor.parameters(): p.requires_grad = False
        for layer in self.wav2vec2.encoder.layers[:-2]:
            for p in layer.parameters(): p.requires_grad = False
        self.classifier = nn.Sequential(
            nn.LayerNorm(768),
            nn.Linear(768, 256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, 2)
        )
        self.loss_fn = nn.CrossEntropyLoss()
    def forward(self, input_values, labels=None):
        out = self.wav2vec2(input_values=input_values)
        pooled = out.last_hidden_state.mean(dim=1)
        logits = self.classifier(pooled)
        probs = torch.softmax(logits, dim=-1)
        res = {"logits": logits, "probs": probs}
        if labels is not None:
            res["loss"] = self.loss_fn(logits, labels)
        return res

model = Classifier().to(device)
best_checkpoint = MODELS_DIR / "best_wav2vec2_deepfake_model.pth"
print(f"Model initialized. Checkpoint target: {best_checkpoint}")


Loading weights: 100%|██████████| 211/211 [00:00<00:00, 28693.65it/s]
[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_hid.bias             | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model initialized. Checkpoint target: e:\Deepfake\models\best_wav2vec2_deepfake_model.pth
